# DINO Feature Extraction — TinyImageNet

Extracts DINO ViT-S/8 embeddings for the TinyImageNet train and val splits.

Output files (saved to Google Drive):
- `tinyimagenet_dino_train_embeddings.pt` — dict with keys `features` (100000, 384) and `labels` (100000,)
- `tinyimagenet_dino_val_embeddings.pt`   — dict with keys `features` (10000, 384)  and `labels` (10000,)

Once downloaded, place them at:
```
embeddings/tinyimagenet/tinyimagenet_dino_train_embeddings.pt
embeddings/tinyimagenet/tinyimagenet_dino_val_embeddings.pt
```

**Note on val restructuring:** TinyImageNet's val split ships as a flat folder with an
annotations file. This notebook restructures it into class subfolders so that
`ImageFolder` can load it correctly. This is done once in the setup cell.

In [ ]:
# Mount Google Drive so we can save large files persistently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 128
DATA_DIR   = '/tmp/tiny-imagenet-200'        # download here — avoids Colab disk quota
SAVE_DIR   = '/content/drive/MyDrive/FYP'   # change if your Drive folder is different

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Device : {DEVICE}')
print(f'Saving to: {SAVE_DIR}')

In [ ]:
# Download and unzip TinyImageNet
# ~236 MB zip, extracts to ~500 MB
import zipfile

ZIP_PATH = '/tmp/tiny-imagenet-200.zip'

if not os.path.exists(DATA_DIR):
    print('Downloading TinyImageNet...')
    os.system(f'wget -q -O {ZIP_PATH} http://cs231n.stanford.edu/tiny-imagenet-200.zip')
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/tmp')
    print(f'Extracted to {DATA_DIR}')
else:
    print('TinyImageNet already present, skipping download.')

In [ ]:
# Restructure the val split so ImageFolder can load it
#
# Original layout (unusable by ImageFolder):
#   val/images/val_0.JPEG, val_1.JPEG, ...
#   val/val_annotations.txt  (image_name  class_id  x  y  w  h)
#
# Target layout (required by ImageFolder):
#   val/n01443537/val_0.JPEG
#   val/n01629819/val_1.JPEG
#   ...

import shutil

VAL_DIR        = os.path.join(DATA_DIR, 'val')
VAL_IMAGES_DIR = os.path.join(VAL_DIR, 'images')
VAL_ANNOT_FILE = os.path.join(VAL_DIR, 'val_annotations.txt')
RESTRUCTURE_FLAG = os.path.join(VAL_DIR, '.restructured')

if not os.path.exists(RESTRUCTURE_FLAG):
    print('Restructuring val split...')

    # Parse annotations: image_name → class_id
    img_to_class = {}
    with open(VAL_ANNOT_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            img_to_class[parts[0]] = parts[1]  # e.g. val_0.JPEG → n01443537

    # Move each image into its class subfolder
    for img_name, class_id in img_to_class.items():
        class_dir = os.path.join(VAL_DIR, class_id)
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(VAL_IMAGES_DIR, img_name)
        dst = os.path.join(class_dir, img_name)
        if os.path.exists(src):
            shutil.move(src, dst)

    # Remove now-empty images/ folder
    if os.path.exists(VAL_IMAGES_DIR):
        shutil.rmtree(VAL_IMAGES_DIR)

    # Write flag so we don't repeat this on reruns
    open(RESTRUCTURE_FLAG, 'w').close()
    print(f'Done. Val split restructured into {len(img_to_class)} class folders.')
else:
    print('Val split already restructured, skipping.')

In [ ]:
# Standard DINO normalisation — must match what the pipeline uses
# Resize to 224 so patches are meaningful; TinyImageNet native res is 64x64
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Load DINO ViT-S/8 — frozen, eval mode
print('Loading DINO ViT-S/8...')
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
model.to(DEVICE)
model.eval()
for param in model.parameters():
    param.requires_grad = False
print('DINO loaded and frozen.')

In [ ]:
def extract_embeddings(split_dir: str, split_name: str) -> dict:
    """
    Extracts L2-normalised DINO embeddings for one TinyImageNet split.

    Args:
        split_dir:  Absolute path to the split folder (must be ImageFolder-compatible)
        split_name: Human-readable name for logging ('train' or 'val')

    Returns:
        dict with keys:
            'features': torch.Tensor (N, 384) — L2-normalised DINO embeddings
            'labels':   torch.Tensor (N,)     — integer class indices (0–199)
    """
    print(f'\nExtracting {split_name} embeddings from {split_dir}...')

    dataset = datasets.ImageFolder(root=split_dir, transform=transform)
    print(f'  Classes: {len(dataset.classes)}, Images: {len(dataset)}')

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,    # must be False — order must match label array
        num_workers=4,
        pin_memory=True
    )

    all_features = []
    all_labels   = []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(DEVICE)

        with torch.no_grad():
            features = model(images)                        # (B, 384)
        features = F.normalize(features, p=2, dim=1)       # L2 normalise

        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

        if (batch_idx + 1) % 10 == 0:
            processed = (batch_idx + 1) * BATCH_SIZE
            print(f'  {processed} / {len(dataset)} images processed...')

    features = torch.cat(all_features, dim=0)   # (N, 384)
    labels   = torch.cat(all_labels,   dim=0)   # (N,)

    print(f'Done. features={features.shape}, labels={labels.shape}')
    return {'features': features, 'labels': labels}

In [ ]:
# Extract train split (100,000 images, 200 classes)
train_dir  = os.path.join(DATA_DIR, 'train')
train_data = extract_embeddings(train_dir, 'train')

train_path = os.path.join(SAVE_DIR, 'tinyimagenet_dino_train_embeddings.pt')
torch.save(train_data, train_path)
print(f'Saved: {train_path}')

In [ ]:
# Extract val split (10,000 images, 200 classes)
val_dir  = os.path.join(DATA_DIR, 'val')
val_data = extract_embeddings(val_dir, 'val')

val_path = os.path.join(SAVE_DIR, 'tinyimagenet_dino_val_embeddings.pt')
torch.save(val_data, val_path)
print(f'Saved: {val_path}')

In [ ]:
# Verification — confirm shapes and that no NaN values crept in
for name, path in [
    ('train', train_path),
    ('val',   val_path)
]:
    data = torch.load(path, weights_only=True)
    f, l = data['features'], data['labels']
    norms = f.norm(dim=1)
    print(f'{name}: features={f.shape}, labels={l.shape}, '
          f'norm_mean={norms.mean():.4f} (should be ~1.0), '
          f'has_nan={f.isnan().any().item()}, '
          f'unique_classes={l.unique().numel()}')

## Next steps

Download both `.pt` files from your Drive and place them at:
```
Image-clustering-FYP/src/embeddings/tinyimagenet/tinyimagenet_dino_train_embeddings.pt
Image-clustering-FYP/src/embeddings/tinyimagenet/tinyimagenet_dino_val_embeddings.pt
```

**Expected shapes:**
- Train: `features=(100000, 384)`, `labels=(100000,)`, `unique_classes=200`
- Val:   `features=(10000, 384)`,  `labels=(10000,)`,  `unique_classes=200`

**Expected runtime on Colab T4/P100:**
- Train: ~12–15 minutes
- Val:   ~1–2 minutes